In [12]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from itertools import combinations

In [13]:
halfcheetah = {
    "SAC": pd.read_csv("HalfCheetah/SAC.csv")["env: HALFCHEETAH - ep_info/Final_position"].values,
    "UDC": pd.read_csv("HalfCheetah/UDC.csv")["env: HALFCHEETAH - ep_info/Final_position"].values,
    "DyLam": pd.read_csv("HalfCheetah/DyLam.csv")["env: HALFCHEETAH - ep_info/Final_position"].values,
}


In [14]:
chicken_banana_all = pd.read_csv("ChickenBanana/rew-total.csv")
setups = ["Baseline", "Decq", "Udc", "Dylam"]
name_map = {
    "Baseline": "Q-Learning",
    "Decq": "Q-Decomp",
    "Udc": "UDC",
    "Dylam": "DyLam",
}
chicken_banana = {
    name_map[setup]: chicken_banana_all[f"setup: {setup} - ep_info/total"].values
    for setup in setups
}

In [15]:
vss = {
    "SAC": pd.read_csv("VSS/SAC.csv")["env: VSS - ep_info/Goal"].values,
    "UDC": pd.read_csv("VSS/UDC.csv")["env: VSS - ep_info/Goal"].values,
    "Tuned-UDC": pd.read_csv("VSS/Tuned-UDC.csv")["env: VSS - ep_info/Goal"].values,
    "DyLam": pd.read_csv("VSS/DyLam.csv")["env: VSS - ep_info/Goal"].values,
}

In [16]:
results = {
    "HalfCheetah": halfcheetah,
    "Chicken-Banana": chicken_banana,
    "VSS": vss,
}

# For each environment, in each setup, we left only the last 10% of the data points
for env, setups in results.items():
    for setup, data in setups.items():
        n = len(data)
        results[env][setup] = data[int(n * 0.9) :]

ALPHA = 0.05
REFERENCE = "DyLam"  # we always compare DyLam vs. each baseline

In [17]:
print("=" * 60)
print(f"Mann-Whitney U tests (two-sided, α={ALPHA})")
print(f"Reference method: {REFERENCE}")
print("=" * 60)

latex_rows = []

for env_name, methods in results.items():
    print(f"\n--- {env_name} ---")
    dylam_scores = methods[REFERENCE]

    for baseline_name, baseline_scores in methods.items():
        if baseline_name == REFERENCE:
            continue

        stat, p = mannwhitneyu(dylam_scores, baseline_scores, alternative="two-sided")
        significant = p < ALPHA
        direction = ">" if np.median(dylam_scores) > np.median(baseline_scores) else "<"

        print(
            f"  DyLam {direction} {baseline_name:12s}  |  "
            f"U={stat:.0f}  p={p:.4f}  "
            f"{'*SIGNIFICANT*' if significant else ''}"
        )

        # For the LaTeX table
        marker = r"$^{*}$" if significant else ""
        latex_rows.append(
            f"  {env_name} & {baseline_name} & "
            f"{np.mean(dylam_scores):.3f} & "
            f"{np.mean(baseline_scores):.3f} & "
            f"{p:.4f}{marker} \\\\"
        )


Mann-Whitney U tests (two-sided, α=0.05)
Reference method: DyLam

--- HalfCheetah ---
  DyLam > SAC           |  U=2452  p=0.0000  *SIGNIFICANT*
  DyLam > UDC           |  U=2500  p=0.0000  *SIGNIFICANT*

--- Chicken-Banana ---
  DyLam > Q-Learning    |  U=40401  p=0.0000  *SIGNIFICANT*
  DyLam > Q-Decomp      |  U=40401  p=0.0000  *SIGNIFICANT*
  DyLam > UDC           |  U=40401  p=0.0000  *SIGNIFICANT*

--- VSS ---
  DyLam > SAC           |  U=20285910  p=0.0000  *SIGNIFICANT*
  DyLam > UDC           |  U=20586242  p=0.0000  *SIGNIFICANT*
  DyLam > Tuned-UDC     |  U=22155928  p=0.0000  *SIGNIFICANT*


In [18]:

# ------------------------------------------------------------------
# LaTeX table (paste into your paper)
# ------------------------------------------------------------------

print("\n\n% ---- LaTeX table ----")
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Pairwise Mann-Whitney U tests: DyLam vs.\ baselines.")
print(r"  $^{*}$ denotes $p < 0.05$ (two-sided).}")
print(r"\label{tab:significance}")
print(r"\begin{tabular}{llccc}")
print(r"\toprule")
print(r"Environment & Baseline & DyLam mean & Baseline mean & $p$-value \\")
print(r"\midrule")
for row in latex_rows:
    print(row)
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


# ------------------------------------------------------------------
# How to extract "one value per seed" from your training curves
# ------------------------------------------------------------------
# If your data is shape (n_seeds, n_steps), do:
#
#   last_10pct = int(0.9 * n_steps)
#   summary = data[:, last_10pct:].mean(axis=1)  # shape (n_seeds,)
#
# That gives you one scalar per seed to feed into the arrays above.
# ------------------------------------------------------------------



% ---- LaTeX table ----
\begin{table}[ht]
\centering
\caption{Pairwise Mann-Whitney U tests: DyLam vs.\ baselines.
  $^{*}$ denotes $p < 0.05$ (two-sided).}
\label{tab:significance}
\begin{tabular}{llccc}
\toprule
Environment & Baseline & DyLam mean & Baseline mean & $p$-value \\
\midrule
  HalfCheetah & SAC & 444.933 & 392.227 & 0.0000$^{*}$ \\
  HalfCheetah & UDC & 444.933 & 324.704 & 0.0000$^{*}$ \\
  Chicken-Banana & Q-Learning & 199.782 & 127.000 & 0.0000$^{*}$ \\
  Chicken-Banana & Q-Decomp & 199.782 & 90.149 & 0.0000$^{*}$ \\
  Chicken-Banana & UDC & 199.782 & 129.985 & 0.0000$^{*}$ \\
  VSS & SAC & 0.855 & 0.078 & 0.0000$^{*}$ \\
  VSS & UDC & 0.855 & 0.057 & 0.0000$^{*}$ \\
  VSS & Tuned-UDC & 0.855 & 0.441 & 0.0000$^{*}$ \\
\bottomrule
\end{tabular}
\end{table}
